# Canada Climate Analysis

Temperature trend analysis and radiative forcing for Canada (1960–2024).

**Data sources:**
- Temperature: Berkeley Earth country time series (`canada-TAVG-monthly.txt`)
- CO₂: NOAA Global Monitoring Laboratory (`co2_annmean_gl.txt`)

**Methods:** Monthly anomaly processing · 11-year running mean · Decadal averages · Linear regression · Stefan-Boltzmann radiative forcing

## 1. Data Loading and Processing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import mean_squared_error

# Load full temperature dataset (Berkeley Earth fixed-width format)
canada_data = pd.read_fwf(
    "canada-TAVG-monthly.txt",
    skiprows=96,
    widths=[6, 6, 10],
    index_col="Year",
    names=["Year", "Month", "Anomaly"]
)

# Subset from 1960 onward
subset_canada = pd.read_fwf(
    "canada-TAVG-monthly.txt",
    skiprows=1325,
    widths=[6, 6, 10],
    index_col="Year",
    names=["Year", "Month", "Anomaly"]
)

display(subset_canada.head())

In [ ]:
# Summary statistics
max_anomaly = subset_canada["Anomaly"].max()
min_anomaly = subset_canada["Anomaly"].min()
avg_anomaly = subset_canada["Anomaly"].mean()
missing_numbers = canada_data["Anomaly"].isna().sum()

print(f"Max anomaly:     {max_anomaly:.2f}°C")
print(f"Min anomaly:     {min_anomaly:.2f}°C")
print(f"Mean anomaly:    {avg_anomaly:.2f}°C")
print(f"Missing values:  {missing_numbers}")

In [ ]:
# Convert monthly anomalies to annual, exclude incomplete 2025
annual_anomalies = subset_canada.groupby("Year")[["Anomaly"]].sum()
annual_anomalies = annual_anomalies[annual_anomalies.index < 2025]

# Convert to absolute temperature using Berkeley Earth baseline (1951-1980)
baseline = -4.86  # Canada reference temperature (°C)
annual_anomalies["Absolute_Temperature"] = annual_anomalies["Anomaly"] + baseline

display(annual_anomalies.head())

## 2. Temperature Time Series

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(
    annual_anomalies.index,
    annual_anomalies["Absolute_Temperature"],
    linewidth=1.5, color='blue'
)
plt.title("Canada Annual Absolute Temperature (1960–2024)")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Load NOAA global CO₂ concentration data
CO2_data = pd.read_fwf(
    "co2_annmean_gl.txt",
    skiprows=39,
    widths=[6, 9],
    names=["Year", "mean"],
    index_col="Year"
)

# Merge temperature and CO₂ datasets on Year
Temp_vs_CO2 = pd.merge(annual_anomalies, CO2_data, on="Year")
display(Temp_vs_CO2.head())

In [ ]:
# Temperature vs CO₂ scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(Temp_vs_CO2["Absolute_Temperature"], Temp_vs_CO2["mean"], alpha=0.7)
plt.xlabel("Absolute Temperature (°C)")
plt.ylabel("CO₂ Concentration (ppm)")
plt.title("Canada Temperature vs Global CO₂ Concentrations")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Trend Analysis

In [ ]:
# 11-year centred running mean (excludes first and last 5 years)
temperatures = pd.DataFrame({"temperatures": annual_anomalies["Absolute_Temperature"]})
df_1960 = pd.DataFrame(
    {"11yr_running_mean": temperatures["temperatures"].rolling(window=11, center=True).mean()},
    index=range(1960, 2025)
)
display(df_1960[5:-5])

In [ ]:
# Decadal averages
annual_anomalies["Decade"] = (annual_anomalies.index // 10) * 10
decadal_avg = annual_anomalies.groupby("Decade")["Absolute_Temperature"].mean().reset_index()
display(decadal_avg)

In [ ]:
# Linear regression (1960–2024)
df_average = pd.DataFrame({"Temperature": annual_anomalies["Absolute_Temperature"]})
df_clean = df_average.dropna()

slope, intercept, r_value, _, _ = stats.linregress(df_clean.index, df_clean["Temperature"])

years = np.arange(1960, 2025)
predicted_values = slope * years + intercept
predicted_2050 = slope * 2050 + intercept

# Metrics
actual_values = df_clean["Temperature"].values
predicted_filtered = slope * np.array(df_clean.index) + intercept
rmse = np.sqrt(mean_squared_error(actual_values, predicted_filtered))
nrmse = rmse / (actual_values.max() - actual_values.min())
pbias = np.mean((predicted_filtered - actual_values) / actual_values) * 100

print(f"R²:                {r_value**2:.3f}")
print(f"Predicted 2050:    {predicted_2050:.2f}°C")
print(f"PBIAS:             {pbias:.2f}%")
print(f"NRMSE:             {nrmse:.4f}")

In [ ]:
# Combined trend plot
df_average["11yr_running_mean"] = df_average["Temperature"].rolling(11, center=True).mean()

x_ref = 1965
y_ref = decadal_avg.loc[decadal_avg["Decade"] == 1960, "Absolute_Temperature"].values[0]
x_recent = 2015
y_recent = decadal_avg.loc[decadal_avg["Decade"] == 2010, "Absolute_Temperature"].values[0]
slope_best = (y_recent - y_ref) / (x_recent - x_ref)
intercept_best = y_ref - slope_best * x_ref

x_range = [df_average.index.min(), df_average.index.max()]

plt.figure(figsize=(12, 8))
plt.plot(df_average.index, df_average["Temperature"], 'gray', alpha=0.6, label="Annual data")
plt.plot(df_average.index, df_average["11yr_running_mean"], 'blue', linewidth=2, label="11-year running mean")
plt.scatter(decadal_avg["Decade"] + 5, decadal_avg["Absolute_Temperature"], c="red", s=50, label="Decadal averages")
plt.plot(x_range, slope_best * np.array(x_range) + intercept_best, 'g--', linewidth=2, label="Best decadal fit (1960s–2010s)")
plt.plot(x_range, intercept + slope * np.array(x_range), 'orange', linestyle='-.', linewidth=2, label=f"Linear regression (R²={r_value**2:.2f})")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.title("Canada Temperature Trends 1960–2024")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Radiative Forcing Analysis

Absorbed fraction $f$ computed using the Stefan-Boltzmann relation:

$$f = 2 - \frac{C_{\text{sol}}(1 - A)}{2 \sigma T^4}$$

Radiative forcing over 1960–2024:

$$\Delta F = \Delta T_E \cdot 4 \left(1 - \frac{f}{2}\right) \sigma T_E^3$$

In [ ]:
# Constants
C_sol = 1361       # Solar constant (W/m²)
sigma = 5.67e-8    # Stefan-Boltzmann constant (W/m²/K⁴)

# Albedo range 0.20 to 0.40 in steps of 0.025
albedos = [round(0.20 + i * 0.025, 3) for i in range(int((0.40 - 0.20) / 0.025) + 1)]

# Temperatures in Kelvin
temps_C = annual_anomalies.loc[1960:2024, "Absolute_Temperature"].tolist()
temps_K = [t + 273.15 for t in temps_C]

# Compute absorbed fraction for each (T, A) combination
data = []
for T in temps_K:
    row = [2 - (C_sol * (1 - A)) / (2 * sigma * T**4) for A in albedos]
    data.append(row)

df_fractions = pd.DataFrame(data, columns=albedos, index=range(1960, 2025))
df_fractions["Yearly Temperature"] = annual_anomalies.loc[1960:2024, "Absolute_Temperature"]
df_fractions.index.name = "Year"

# Reorder columns
cols = ["Yearly Temperature"] + albedos
df_fractions = df_fractions[cols]
display(df_fractions.head())

In [ ]:
# Cap absorbed fractions at 1
df_fractions2 = df_fractions.drop("Yearly Temperature", axis=1).clip(upper=1)
display(df_fractions2.head())

In [ ]:
# Radiative forcing per albedo value
delta_T_E = temps_K[-1] - temps_K[0]  # Temperature change over full interval (K)
T_start = temps_K[0]                   # Reference temperature (start of interval)

radiative_forcing = delta_T_E * 4 * (1 - df_fractions2.mean() / 2) * sigma * T_start**3

rf_table = pd.DataFrame({"Albedo": albedos, "Radiative Forcing (W/m²)": radiative_forcing.values})
display(rf_table)